# QM 640 Capstone — Step 3e2: First-Pass Classification Suggestions

This notebook does **not** make final classification decisions - it narrows a large candidate pool down to a manageable review list, using the same word-boundary-corrected keyword logic as `03g_reclassification_review.ipynb` (kept identical here so both notebooks agree; if you ever edit the keyword lists, update both).

**Two-tier approach:**
- Rows with **zero** AI-keyword matches get auto-set to `is_genuine_ai_event = N` directly, with the reason logged in `exclude_reason` - defensible because Step 1's EDGAR search already pre-filtered on AI-specific phrases, so a genuine miss here is low-risk, and it's fully auditable (every auto-exclusion is logged, not silent).
- Every row with **any** AI-keyword match gets a `suggested_is_genuine_ai_event` / `suggested_announcement_type` + `matched_keywords` column to speed up your read, but `is_genuine_ai_event` / `announcement_type` themselves are left **blank** - you confirm or override every one by hand, reading `text_snippet`. No auto-fill for `Y` rows, ever.

**Run this after `03e_fetch_snippets.ipynb`, before `03f_merge_review.ipynb`.** Does not touch `screening_recode_sample.csv` - that file must stay blind for the independent recoder, no suggestions of any kind.

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every other cell in this notebook reads from and writes to.

In [1]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 943, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 943 (delta 56), reused 89 (delta 33), pack-reused 813 (from 1)
Receiving objects: 100% (943/943), 7.14 MiB | 7.94 MiB/s, done.
Resolving deltas: 100% (497/497), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [2]:
!pip install -q pandas

## Cell 3 — Configuration

In [3]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
REVIEW_FILE = os.path.join(RAW_DIR, "screening_TO_REVIEW.csv")

## Cell 4 — Corrected classifier (word-boundary matching)

Identical logic to `03g_reclassification_review.ipynb`'s Cell 4 - `"ai"` only matches the standalone word "AI," not the substring inside "maintain," "certain," "available," "FAILED," etc. Also flags **low-confidence** matches - a hit on only a single generic keyword (bare "ai", or an M&A keyword like "purchase"/"agreement" that's common in unrelated 8-K boilerplate) - as worth extra scrutiny even when it does suggest `Y`.

In [4]:
import re
import pandas as pd

AI_KEYWORDS = [
    "artificial intelligence", "ai", "generative ai", "genai", "machine learning",
    "deep learning", "large language model", "language model", "foundation model", "llm",
    "gpt", "copilot", "neural network", "computer vision", "predictive ai", "autonomous ai",
    "ai assistant", "chatbot", "ai platform", "ai software", "ai infrastructure", "ai chip",
    "ai accelerator",
]
PARTNERSHIP_KEYWORDS = [
    "partner", "partners", "partnered", "partnership", "collaborate", "collaboration",
    "alliance", "strategic alliance", "joint venture", "jointly", "agreement",
    "memorandum of understanding", "mou", "teamed up", "working with",
]
RD_KEYWORDS = [
    "launch", "launched", "introduce", "introduced", "release", "released", "develop",
    "developed", "developing", "research", "innovation", "innovative", "prototype", "patent",
    "roadmap", "platform", "solution", "tool", "engine", "framework", "service", "capability",
    "model", "assistant", "copilot", "training", "inference", "gpu", "chip", "processor",
]
MA_KEYWORDS = [
    "acquire", "acquired", "acquisition", "buy", "bought", "purchase", "purchased",
    "takeover", "merge", "merged", "merger", "invests in", "investment in", "stake in",
]

# Keywords too generic to trust on their own, even with word boundaries - a hit on
# ONLY these (no more specific keyword also matching) gets flagged as low-confidence.
GENERIC_AI_KEYWORDS = {"ai"}
GENERIC_MA_KEYWORDS = {"purchase", "purchased", "buy", "bought", "agreement", "stake in"}


def preprocess(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text).lower())


def matched_keywords(text, keywords):
    """Word-boundary match - returns the list of keywords that actually hit."""
    return [k for k in keywords if re.search(rf"\b{re.escape(k)}\b", text)]


def classify_fixed(snippet):
    text = preprocess(snippet)
    if text == "":
        return "N", "", [], False

    ai_hits = matched_keywords(text, AI_KEYWORDS)
    if not ai_hits:
        return "N", "", [], False

    low_confidence = set(ai_hits).issubset(GENERIC_AI_KEYWORDS)

    ma_hits = matched_keywords(text, MA_KEYWORDS)
    if ma_hits:
        if set(ma_hits).issubset(GENERIC_MA_KEYWORDS):
            low_confidence = True
        return "Y", "M&A", ai_hits + ma_hits, low_confidence

    partner_hits = matched_keywords(text, PARTNERSHIP_KEYWORDS)
    if partner_hits:
        return "Y", "partnership", ai_hits + partner_hits, low_confidence

    rd_hits = matched_keywords(text, RD_KEYWORDS)
    if rd_hits:
        return "Y", "R&D", ai_hits + rd_hits, low_confidence

    # AI mentioned, no more specific type keyword matched - low confidence by definition
    return "Y", "R&D", ai_hits, True


print("Corrected classifier loaded.")

Corrected classifier loaded.


## Cell 5 — Apply: auto-exclude zero-match rows, suggest for everything else

In [5]:
review = pd.read_csv(REVIEW_FILE)

# Fail loud instead of silently classifying duplicated rows
assert review["accession_no"].is_unique, (
    f"DUPLICATE accession_no in {REVIEW_FILE} "
    f"({len(review)} rows, {review['accession_no'].nunique()} unique). "
    f"Run 03h_health_check.ipynb before continuing."
)
print(f"Loaded {len(review)} rows from {REVIEW_FILE}")

results = review["text_snippet"].apply(classify_fixed)
review["suggested_is_genuine_ai_event"] = results.apply(lambda r: r[0])
review["suggested_announcement_type"] = results.apply(lambda r: r[1])
review["matched_keywords"] = results.apply(lambda r: ", ".join(r[2]))
review["low_confidence_match"] = results.apply(lambda r: r[3])

zero_match = review["suggested_is_genuine_ai_event"] == "N"

# Auto-exclude zero-match rows directly - documented, auditable, not silent
if "exclude_reason" not in review.columns:
    review["exclude_reason"] = ""
review.loc[zero_match, "is_genuine_ai_event"] = "N"
review.loc[zero_match, "announcement_type"] = ""

def append_reason(existing, addition):
    existing = "" if pd.isna(existing) else str(existing).strip()
    return f"{existing}; {addition}" if existing else addition

review.loc[zero_match, "exclude_reason"] = review.loc[zero_match, "exclude_reason"].apply(
    lambda x: append_reason(x, "no AI-related keyword match in text_snippet (auto-excluded, Step 3e2)")
)

# Everything else: leave is_genuine_ai_event / announcement_type BLANK for human review,
# suggestion columns are there to speed up reading, not to replace judgment
needs_review = ~zero_match

n_total = len(review)
n_auto_excluded = zero_match.sum()
n_needs_review = needs_review.sum()
n_low_conf = (needs_review & review["low_confidence_match"]).sum()

print(f"\nTotal candidate rows: {n_total}")
print(f"Auto-excluded (no AI keyword match at all): {n_auto_excluded} ({n_auto_excluded/n_total:.1%})")
print(f"Needs human review (has at least one AI keyword match): {n_needs_review} ({n_needs_review/n_total:.1%})")
print(f"  of which low-confidence (single generic keyword only): {n_low_conf}")
print(f"\nSuggested type breakdown among rows needing review:")
print(review.loc[needs_review, "suggested_announcement_type"].value_counts())

Loaded 814 rows from /content/QM640-WALSH-CAPSTONE/data/raw/screening_TO_REVIEW.csv

Total candidate rows: 814
Auto-excluded (no AI keyword match at all): 443 (54.4%)
Needs human review (has at least one AI keyword match): 371 (45.6%)
  of which low-confidence (single generic keyword only): 261

Suggested type breakdown among rows needing review:
suggested_announcement_type
R&D            167
M&A            143
partnership     61
Name: count, dtype: int64


/tmp/ipykernel_1326/1424332360.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review.loc[zero_match, "is_genuine_ai_event"] = "N"
/tmp/ipykernel_1326/1424332360.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review.loc[zero_match, "announcement_type"] = ""


## Cell 6 — Save, reordered for review

In [6]:
review_cols = [
    "accession_no", "company_name", "file_date", "text_snippet",
    "suggested_is_genuine_ai_event", "suggested_announcement_type",
    "matched_keywords", "low_confidence_match",
    "is_genuine_ai_event", "announcement_type",   # blank for Y-candidates - fill these in by hand
    "item_codes", "item_in_scope", "exclude_reason", "filing_url",
]
review = review[[c for c in review_cols if c in review.columns]]
import csv
review.to_csv(REVIEW_FILE, index=False, quoting=csv.QUOTE_ALL)  # robust against Excel/Sheets CSV round-trip corruption
print(f"Saved -> {REVIEW_FILE}")
review[review["is_genuine_ai_event"] != "N"].head()

Saved -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_TO_REVIEW.csv


,accession_no,company_name,file_date,text_snippet,suggested_is_genuine_ai_event,suggested_announcement_type,matched_keywords,low_confidence_match,is_genuine_ai_event,announcement_type,item_codes,item_in_scope,exclude_reason,filing_url
0,0001652044-23-000013:googexhibit991q42022.htm,"Alphabet Inc. (GOOG, GOOGL) (CIK 0001652044)",2023-02-02,EX-99.1 2 googexhibit991q42022.htm EX-99.1 Doc...,Y,R&D,ai,True,NaN,NaN,NaN,REVIEW,,https://www.sec.gov/cgi-bin/browse-edgar?actio...
8,0001326801-23-000063:meta03312023-exhibit991.htm,"Meta Platforms, Inc. (META) (CIK 0001326801)",2023-04-26,EX-99.1 2 meta03312023-exhibit991.htm EX-99.1 ...,Y,R&D,ai,True,NaN,NaN,NaN,REVIEW,,https://www.sec.gov/cgi-bin/browse-edgar?actio...
14,0001199835-23-000308:jewl_ex99-1.htm,Adamas One Corp. (JEWL) (CIK 0001884072),2023-06-05,EX-99.1 3 jewl_ex99-1.htm PRESS RELEASE DATED ...,Y,M&A,"artificial intelligence, ai, acquisition, stak...",False,NaN,NaN,"1.01,3.02,8.01,9.01",Y,,https://www.sec.gov/cgi-bin/browse-edgar?actio...
16,0001628280-23-025834:exhibit991-reversestocksp...,"Ontrak, Inc. (OTRK, OTRKP) (CIK 0001136174)",2023-07-27,EX-99.1 3 exhibit991-reversestockspl.htm EX-99...,Y,R&D,ai,True,NaN,NaN,"8.01,9.01",Y,,https://www.sec.gov/cgi-bin/browse-edgar?actio...
19,0001477932-23-006236:haia_ex31.htm,"Healthcare AI Acquisition Corp. (HAIA, HAIAU,...",2023-08-17,EX-3.1 2 haia_ex31.htm AMENDMENT haia_ex31.htm...,Y,M&A,"ai, acquisition",True,NaN,NaN,"5.03,5.07,8.01,9.01",Y,,https://www.sec.gov/cgi-bin/browse-edgar?actio...


## Commit and push results back to GitHub

In [7]:
!git -C {BASE_DIR} add "data/raw/screening_TO_REVIEW.csv"
!git -C {BASE_DIR} commit -m "Step 3e2: auto-exclude zero-match rows, add classification suggestions for the rest"
!git -C {BASE_DIR} push

[main 524ee3c] Step 3e2: auto-exclude zero-match rows, add classification suggestions for the rest
 1 file changed, 815 insertions(+), 815 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 11.78 KiB | 754.00 KiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   cd0e5a4..524ee3c  main -> main


## >>> STOP HERE — manual step, outside this notebook <<<

Open `screening_TO_REVIEW.csv`. Rows already marked `is_genuine_ai_event = N` are auto-excluded and logged - spot-check a handful if you want reassurance, but they don't need full review.

For every remaining row: read `text_snippet` (click `filing_url` only if it's ambiguous or cut off), glance at `suggested_is_genuine_ai_event` / `suggested_announcement_type` / `matched_keywords` as a starting point, then fill in your own `is_genuine_ai_event` (Y/N) and `announcement_type` (partnership/R&D/M&A) - agree with the suggestion or override it, but make the call yourself. Give extra scrutiny to rows where `low_confidence_match` is `True`.

Push the completed file, then run `03f_merge_review.ipynb`, followed separately by an independent pass on `screening_recode_sample.csv` and `03`'s Part B (Cohen's kappa).